In [ ]:






# Fetch league leader data for batting average
beans = statsapi.league_leader_data('battingAverage', season=2025, limit=50, statGroup='hitting')
stats = []  # List to store formatted stats
names = []  # List to store player names





for x in beans:  # Iterate through the league leader data
    if x[2] in mlb_games_today:  # Check if the player's team is playing today
        stringy = str(x[1]) + "-" + str(x[2]) + "-AVG 2025-" + str(x[3])  # Format the stat
        stats.append(stringy)  # Add the formatted stat to the list
        names.append(x[1])  # Add the player's name to the list



# Collect stats for other categories and years
categories = [  # List of stat categories to fetch
    ('battingAverage', 'AVG'),
    ('homeRuns', 'HR'),
    ('gamesPlayed', 'aGP'),
    ('onBasePlusSlugging', 'OBPS'),
    ('strikeOuts', 'SO'),
    ('hits', 'HITS'),
]
# years = [2025, 2024]  # Uncomment to include multiple years
years = [2025]  # List of years to fetch stats for

for category, label in categories:  # Iterate through each category
    for year in years:  # Iterate through each year
        beans = statsapi.league_leader_data(category, season=year, limit=75, statGroup='hitting')
        for x in beans:  # Iterate through the league leader data
            if x[1] in names:  # Check if the player's name is in the list
                stringy = f"{x[1]}-{x[2]}-{label} {year}-{x[3]}"  # Format the stat
                stats.append(stringy)  # Add the formatted stat to the list

# Sort stats alphabetically
stats.sort()

# Format the stats for writing to the file
formatted_stats = "\n".join(
    f"{x.split('-')[0]:<20} {x.split('-')[1]:<20} {x.split('-')[2]:<12} {x.split('-')[3]:<5}" for x in stats
)

# Check if the file exists
if os.path.exists(file_name):
    with open(file_name, "r") as file:
        existing_content = file.read()  # Read the existing content

    # Prepend only if new content is not already in the file
    if formatted_stats not in existing_content:
        with open(file_name, "w") as file:
            file.write(formatted_stats + "\n\n" + existing_content)  # Prepend new content
else:
    # Write new content if the file doesn't exist
    with open(file_name, "w") as file:
        file.write(mlb_date + "\n")  # Write the date
        file.write(formatted_stats)  # Write the formatted stats

print(f"Output written to {file_name}")  # Notify the user of successful write


Output written to aLEAGUE LEADERS.txt


In [ ]:
#----
import os

# File path
file_path = "aLEAGUE LEADERS.txt"

# Read the file and parse the data
player_stats = {}
with open(file_path, "r") as file:
    lines = file.readlines()

# Extract the date from the first line
file_date = lines[0].strip()

# Parse the rest of the lines
for line in lines[1:]:
    parts = line.strip().split()
    if len(parts) < 6:
        continue  # Skip invalid lines

    # Extract data
    name = " ".join(parts[:2])  # First and last name
    team = " ".join(parts[2:-3])  # Team name
    stat_type = parts[-3]  # Statistic type (e.g., AVG, HR)
    year = parts[-2]  # Year
    value = parts[-1]  # Statistic value

    # Create a unique key for each player and stat
    key = (stat_type, year)
    if key not in player_stats:
        player_stats[key] = []
    player_stats[key].append((name, team, value))

# Sort the stats alphabetically by player name
sorted_stats = {}
for key, players in player_stats.items():
    sorted_stats[key] = sorted(players, key=lambda x: (x[0], x[1]))

# Prepare the sorted data for prepending
sorted_output = [f"Sorted Stats as of {file_date}:\n"]
for (stat_type, year), players in sorted_stats.items():
    sorted_output.append(f"\n{stat_type} {year}:\n")
    for name, team, value in players:
        sorted_output.append(f"{name:<20} {team:<25} {value}")

# Combine the sorted data into a single string
sorted_output_text = "\n".join(sorted_output)

# Read the original file content
with open(file_path, "r") as file:
    original_content = file.read()

# Prepend the sorted data to the original content
new_content = sorted_output_text + "\n\n" + original_content

# Write the updated content back to the file
with open(file_path, "w") as file:
    file.write(new_content)

print(f"File '{file_path}' has been updated with sorted stats prepended.")
#------



In [ ]:
from collections import defaultdict
import os
from datetime import datetime

# File path
file_path = "aLEAGUE LEADERS.txt"
mlb_date_str = datetime.now().strftime("%m/%d/%Y")

# Data structure to store player stats
player_stats = defaultdict(lambda: defaultdict(list))

# Read the file and parse the data
with open(file_path, "r") as file:
    for line in file:
        parts = line.strip().split()
        if len(parts) < 6:
            continue  # Skip invalid lines
        
        # Extract data
        name = " ".join(parts[:2])  # First and last name
        team = " ".join(parts[2:-3])  # Team name
        name = name + " " + team
        stat_type = parts[-3]  # Statistic type (e.g., AVG, HR)
        year = parts[-2]  # Year
        value = parts[-1]  # Statistic value
        
        # Store the data
        player_stats[name][stat_type].append((year, value))

# Function to calculate a player's performance score (example: based on AVG and OBPS)
def calculate_score(stats):
    avg = max(float(v[1]) for v in stats.get("AVG", [("0", "0")]))
    obps = max(float(v[1]) for v in stats.get("OBPS", [("0", "0")]))
    return avg + obps  # Example scoring formula

# Rank players based on their performance score
ranked_players = sorted(player_stats.items(), key=lambda x: calculate_score(x[1]), reverse=True)

# Prepare the output
line_str = "Top 5 Players: " + mlb_date_str
output_lines = [line_str]
for i, (player, stats) in enumerate(ranked_players[:5], start=1):
    output_lines.append(f"\n{i}. {player}")
    for stat_type, values in stats.items():
        output_lines.append(f"   {stat_type}: {', '.join([f'{v[0]}: {v[1]}' for v in values])}")
output_lines.append("")  # Add a blank line at the end

# File to write the output
output_file = "aTOP LEAGUE LEADERS.txt"

# Prepend the data to the file if it exists
if os.path.exists(output_file):
    with open(output_file, "r") as file:
        existing_content = file.read()
    with open(output_file, "w") as file:
        file.write("\n".join(output_lines) + "\n" + existing_content)
else:
    with open(output_file, "w") as file:
        file.write("\n".join(output_lines) + "\n")

print(f"Output written to {output_file}")